[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Donggyurim/neuroimaging-training/blob/main/notebooks/FRIENDS_sub136_FastSurfer_HypVINN_Colab.ipynb)

# FRIENDS sub-136 FastSurfer + HypVINN (Colab)

This notebook processes **one** BIDS T1w image for `sub-136` with FastSurfer **v2.5.4**, including HypVINN. It runs the complete FastSurfer segmentation pipeline and cortical surface reconstruction, then prepares the output at the FRIENDS derivatives path.

Use **Runtime → Change runtime type → T4 GPU** if available. FastSurfer's surface reconstruction also needs a FreeSurfer license file, which this notebook asks you to upload for the current session only.

**Colab cannot write to Monash M3's `/fs04/scratch2/gz87/FRIENDS` filesystem.** This notebook saves the result and a transfer archive in Google Drive, laid out as `derivatives/fastsurfer/sub-136`. At the end it gives the M3 commands to stage and install the folder. The transfer step is required.

Only process FRIENDS data in Colab if your data governance rules permit using Google Colab and Google Drive for these files. The T1 and FreeSurfer license are uploaded to the temporary Colab runtime; the license is not copied to Drive or included in the archive.

FastSurfer v2.5.4 is pinned because the 2.5.0 release fixed a division-by-zero failure in partial-volume statistics. HypVINN is enabled by default; the command below does not pass `--no_hypothal`.

## 1. Connect Drive and create a run folder

The Drive folder stores the completed subject output, logs, QC images, regional-volume CSV, provenance, and a transfer archive. The working files and uploaded license stay in the Colab runtime.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import datetime, hashlib, json, os, platform, shutil, subprocess, sys, tarfile, urllib.request

drive.mount('/content/drive')

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
DRIVE_ROOT = Path('/content/drive/MyDrive/Neuroimaging-Training')
RESULTS = DRIVE_ROOT / 'results' / 'FRIENDS_sub-136' / RUN_ID
WORK = Path('/content/friends_sub136') / RUN_ID
INPUT_DIR = WORK / 'input'
SUBJECTS_DIR = WORK / 'subjects'
for folder in (RESULTS, INPUT_DIR, SUBJECTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SUBJECT = 'sub-136'
print('Drive results:', RESULTS)
print('Colab working directory:', WORK)

## 2. Upload the sub-136 T1w image and sidecar

Upload the one full-head T1w NIfTI (`sub-136_acq-uniden_T1w.nii.gz`) and, if available, its matching JSON sidecar. FastSurfer does not need the rest of the BIDS participant folder. The JSON lets the notebook use the correct 3T atlas flag when `MagneticFieldStrength` is 3.

In [ ]:
uploaded = files.upload()
t1_candidates = [name for name in uploaded if name.endswith(('_T1w.nii.gz', '_T1w.nii')) and 'sub-136' in name]
assert len(t1_candidates) == 1, f'Upload exactly one sub-136 T1w NIfTI; found: {t1_candidates}'
T1_NAME = t1_candidates[0]
T1 = INPUT_DIR / T1_NAME
T1.write_bytes(uploaded[T1_NAME])

sidecar_name = T1_NAME[:-7] + '.json' if T1_NAME.endswith('.nii.gz') else T1_NAME[:-4] + '.json'
SIDECAR = INPUT_DIR / sidecar_name
if sidecar_name in uploaded:
    SIDECAR.write_bytes(uploaded[sidecar_name])
    sidecar = json.loads(SIDECAR.read_text())
else:
    sidecar = {}
    print('No matching JSON sidecar uploaded; the 3T atlas flag will not be set.')

print('T1:', T1, f'({T1.stat().st_size:,} bytes)')
print('Sidecar:', 'available' if sidecar else 'not uploaded')

## 3. Upload your FreeSurfer license

Upload the FreeSurfer license text file. It is written only to the temporary Colab runtime and is excluded from Drive results and the transfer archive. Obtain the license through your FreeSurfer registration if you do not already have it on the computer running Colab.

In [ ]:
license_upload = files.upload()
assert len(license_upload) == 1, 'Upload exactly one FreeSurfer license file.'
LICENSE_PATH = WORK / 'freesurfer_license.txt'
LICENSE_PATH.write_bytes(next(iter(license_upload.values())))
os.chmod(LICENSE_PATH, 0o600)
assert LICENSE_PATH.stat().st_size > 0, 'The uploaded license file is empty.'
print('License staged in this Colab session only.')

## 4. Install the pinned FastSurfer and FreeSurfer versions

This follows the source-install approach used by the training notebook, with FastSurfer pinned to v2.5.4 and FreeSurfer pinned to v7.4.1. The latter is required for the full surface pipeline. Installation can take several minutes and downloads several gigabytes.

In [ ]:
import re, tomllib

FASTSURFER_TAG = 'v2.5.4'
FS_HOME = Path('/content/FastSurfer')
if not (FS_HOME / '.git').exists():
    subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', FASTSURFER_TAG,
                    'https://github.com/Deep-MI/FastSurfer.git', str(FS_HOME)], check=True)
else:
    subprocess.run(['git', '-C', str(FS_HOME), 'fetch', '--quiet', '--tags'],
                   check=True)
    subprocess.run(['git', '-C', str(FS_HOME), 'checkout', '--quiet', FASTSURFER_TAG],
                   check=True)

with (FS_HOME / 'pyproject.toml').open('rb') as handle:
    project = tomllib.load(handle)
assert project['project']['version'] == '2.5.4', project['project']['version']

# Use the PyTorch versions validated for FastSurfer v2.5.4 and CUDA 12.6.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--index-url', 'https://download.pytorch.org/whl/cu126',
                'torch==2.7.1', 'torchvision==0.22.1'], check=True)

dependencies = project['project']['dependencies']
dependencies = [dep for dep in dependencies
                if re.split(r'[\s<>=!~;\[]', dep, maxsplit=1)[0].lower()
                not in ('torch', 'torchvision', 'pip')]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *dependencies],
               check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
                '--editable', str(FS_HOME)], check=True)

import numpy as np, nibabel as nib, pandas as pd, matplotlib.pyplot as plt, torch
print('FastSurfer:', project['project']['version'])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. The run will use CPU and may take much longer.')
(RESULTS / 'environment.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

In [ ]:
# Install the FreeSurfer version specified by the pinned FastSurfer v2.5.4 source.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'libglu1-mesa', 'libgomp1', 'libxmu6', 'libxt6',
                'libxrender1', 'libxext6'], check=True)

FREESURFER_VERSION = '7.4.1'
FS_TARBALL_URL = (
    f'https://surfer.nmr.mgh.harvard.edu/pub/dist/freesurfer/{FREESURFER_VERSION}/'
    f'freesurfer-linux-ubuntu22_amd64-{FREESURFER_VERSION}.tar.gz'
)
FS_TARBALL = WORK / f'freesurfer-{FREESURFER_VERSION}.tar.gz'
if not FS_TARBALL.exists():
    with urllib.request.urlopen(FS_TARBALL_URL, timeout=120) as response, FS_TARBALL.open('wb') as out:
        shutil.copyfileobj(response, out, length=8 * 1024 * 1024)
assert FS_TARBALL.stat().st_size > 500_000_000, 'FreeSurfer download looks incomplete.'

with tarfile.open(FS_TARBALL, 'r:gz') as archive:
    members = archive.getmembers()
    root = Path('/content').resolve()
    for member in members:
        resolved = (root / member.name).resolve()
        if resolved != root and root not in resolved.parents:
            raise ValueError(f'Unsafe path in FreeSurfer archive: {member.name}')
    archive.extractall(root)

setup_scripts = list(Path('/content').glob('freesurfer*/SetUpFreeSurfer.sh'))
if not setup_scripts:
    setup_scripts = list(Path('/content').glob('freesurfer/**/SetUpFreeSurfer.sh'))
assert setup_scripts, 'FreeSurfer setup script was not found after extraction.'
FREESURFER_HOME = setup_scripts[0].parent
assert (FREESURFER_HOME / 'bin' / 'recon-all').exists(), 'FreeSurfer recon-all is missing.'
print('FreeSurfer home:', FREESURFER_HOME)

## 5. Check the input MRI

The notebook checks that the file is a readable 3D image and shows three central slices before processing. Stop here if the image is not the intended participant or is visibly incomplete.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert T1.is_file() and T1.stat().st_size > 0
image = nib.as_closest_canonical(nib.load(str(T1)))
assert len(image.shape) == 3, f'Expected a 3D T1w image, got shape {image.shape}'
t1_data = image.get_fdata(dtype=np.float32)
assert np.isfinite(t1_data).all(), 'T1w contains non-finite values.'
print('Shape:', image.shape)
print('Voxel size (mm):', nib.affines.voxel_sizes(image.affine))
print('T1 SHA256:', sha256(T1))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
vmax = float(np.percentile(t1_data, 99))
for axis, (ax, title) in enumerate(zip(axes, ('Sagittal', 'Coronal', 'Axial'))):
    ax.imshow(np.take(t1_data, t1_data.shape[axis] // 2, axis=axis).T,
              origin='lower', cmap='gray', vmax=vmax)
    ax.set_title(title)
    ax.axis('off')
fig.tight_layout()
fig.savefig(RESULTS / 'input_preview.png', dpi=150)
plt.show()

## 6. Run FastSurfer with HypVINN enabled

The default segmentation modules include HypVINN. This uses the full `run_fastsurfer.sh` workflow, including surface reconstruction; it does **not** pass `--seg_only` or `--no_hypothal`. Allow the cell to finish and keep the Colab session connected. A GPU runtime is strongly recommended.

In [ ]:
import shlex

THREADS = min(4, os.cpu_count() or 2)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
command = [
    str(FS_HOME / 'run_fastsurfer.sh'),
    '--t1', str(T1),
    '--sid', SUBJECT,
    '--sd', str(SUBJECTS_DIR),
    '--fs_license', str(LICENSE_PATH),
    '--threads', str(THREADS),
    '--parallel',
    '--device', DEVICE,
    '--allow_root',
]
if float(sidecar.get('MagneticFieldStrength', 0) or 0) == 3.0:
    command.append('--3T')

provenance = {
    'participant_id': SUBJECT,
    'input_file': T1_NAME,
    'input_sha256': sha256(T1),
    'sidecar_file': sidecar_name if sidecar else None,
    'magnetic_field_strength_tesla': sidecar.get('MagneticFieldStrength'),
    'fastsurfer_version': FASTSURFER_TAG,
    'freesurfer_version': FREESURFER_VERSION,
    'hypvinn_enabled': True,
    'surface_pipeline_enabled': True,
    'device': DEVICE,
    'threads': THREADS,
    'command': command,
    'run_id': RUN_ID,
    'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'python': platform.python_version(),
    'torch': torch.__version__,
}
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))

env = os.environ.copy()
env.update({
    'FREESURFER_HOME': str(FREESURFER_HOME),
    'FS_LICENSE': str(LICENSE_PATH),
    'SUBJECTS_DIR': str(SUBJECTS_DIR),
    'OMP_NUM_THREADS': str(THREADS),
    'MKL_NUM_THREADS': str(THREADS),
    'OPENBLAS_NUM_THREADS': str(THREADS),
})
shell_command = (
    f'source {shlex.quote(str(FREESURFER_HOME / "SetUpFreeSurfer.sh"))} >/dev/null '
    f'&& exec {shlex.join(command)}'
)
LOG = RESULTS / 'fastsurfer.log'
with LOG.open('w') as log:
    process = subprocess.Popen(['bash', '-lc', shell_command], cwd=FS_HOME, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()

if return_code:
    raise RuntimeError(f'FastSurfer exited with status {return_code}. See {LOG}.')
OUTPUT = SUBJECTS_DIR / SUBJECT
print('FastSurfer process finished:', OUTPUT)

## 7. Verify the main outputs

Check the full segmentation, HypVINN files, and both cortical surface sets before exporting anything. Missing required files stop the notebook.

In [ ]:
SEG = OUTPUT / 'mri/aparc.DKTatlas+aseg.deep.mgz'
CONFORMED = OUTPUT / 'mri/orig.mgz'
HYPO_SEG = OUTPUT / 'mri/hypothalamus.HypVINN.nii.gz'
HYPO_STATS = OUTPUT / 'stats/hypothalamus.HypVINN.stats'
required = [
    SEG, CONFORMED, HYPO_SEG, HYPO_STATS,
    OUTPUT / 'surf/lh.white', OUTPUT / 'surf/rh.white',
    OUTPUT / 'surf/lh.pial', OUTPUT / 'surf/rh.pial',
    OUTPUT / 'surf/lh.thickness', OUTPUT / 'surf/rh.thickness',
]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError('FastSurfer is incomplete; missing or empty outputs:\n' + '\n'.join(missing))
print('Required outputs verified.')
print('HypVINN segmentation:', HYPO_SEG)
print('HypVINN statistics:', HYPO_STATS)
done_marker = OUTPUT / 'scripts/recon-all.done'
print('Completion marker:', 'present' if done_marker.exists() else 'not present; required output files were checked individually')

## 8. Review whole-brain and HypVINN overlays

The main segmentation is overlaid on FastSurfer's matching conformed image. HypVINN is also displayed; if its grid differs, it is nearest-neighbour resampled to the conformed image for this visual check. Inspect the overlays before transferring the result.

In [ ]:
seg_image = nib.as_closest_canonical(nib.load(str(SEG)))
conformed_image = nib.as_closest_canonical(nib.load(str(CONFORMED)))
assert (seg_image.shape == conformed_image.shape
        and np.allclose(seg_image.affine, conformed_image.affine)), 'Whole-brain QC grids do not match.'
seg = np.asarray(seg_image.dataobj).astype(np.int32)
anatomy = conformed_image.get_fdata(dtype=np.float32)
hypo_image = nib.as_closest_canonical(nib.load(str(HYPO_SEG)))
if hypo_image.shape != conformed_image.shape or not np.allclose(hypo_image.affine, conformed_image.affine):
    from nibabel.processing import resample_from_to
    hypo_image = resample_from_to(hypo_image, (conformed_image.shape, conformed_image.affine), order=0)
hypo = np.asarray(hypo_image.dataobj)
assert np.isfinite(seg).all() and np.isfinite(hypo).all()

def show_overlay(labels, axis=2, fraction=0.5, title='FastSurfer'):
    index = min(int(fraction * (labels.shape[axis] - 1)), labels.shape[axis] - 1)
    base = np.take(anatomy, index, axis=axis).T
    layer = np.take(labels, index, axis=axis).T
    mask = layer == 0
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(base, origin='lower', cmap='gray', vmax=np.percentile(anatomy, 99))
    ax.imshow(np.ma.masked_where(mask, layer), origin='lower', cmap='turbo',
              alpha=0.35, interpolation='nearest')
    ax.set_title(f'{title} — slice {index}')
    ax.axis('off')
    return fig

for axis in range(3):
    fig = show_overlay(seg, axis=axis, title='Whole-brain segmentation')
    fig.savefig(RESULTS / f'qc_whole_brain_axis{axis}.png', dpi=150)
    plt.show()

hypo_binary = (hypo > 0).astype(np.uint8)
for axis in (1, 2):
    fig = show_overlay(hypo_binary, axis=axis, title='HypVINN')
    fig.savefig(RESULTS / f'qc_hypvinn_axis{axis}.png', dpi=150)
    plt.show()

## 9. Export regional volumes

The CSV uses hard-label voxel counts multiplied by voxel volume. It is a descriptive export, not the partial-volume-corrected FastSurfer statistics table. FastSurfer and HypVINN `stats` files are retained in the subject output folder.

In [ ]:
lookup = {}
lut_path = FS_HOME / 'FastSurferCNN/config/FreeSurferColorLUT.txt'
for line in lut_path.read_text().splitlines():
    fields = line.split()
    if fields and fields[0].isdigit() and len(fields) >= 2:
        lookup[int(fields[0])] = fields[1]

ids, counts = np.unique(seg, return_counts=True)
voxel_mm3 = float(abs(np.linalg.det(seg_image.affine[:3, :3])))
rows = [
    {
        'participant_id': SUBJECT,
        'label_id': int(label),
        'region': lookup.get(int(label), f'UNMAPPED_{label}'),
        'voxel_count': int(count),
        'volume_mm3': float(count * voxel_mm3),
        'volume_ml': float(count * voxel_mm3 / 1000),
    }
    for label, count in zip(ids, counts) if label != 0
]
table = pd.DataFrame(rows).sort_values('label_id').reset_index(drop=True)
assert np.isclose(table.volume_mm3.sum(), np.count_nonzero(seg) * voxel_mm3)
table.to_csv(RESULTS / 'regional_volumes_hard_label.csv', index=False)
display(table.head(20))
print('Rows exported:', len(table))
print('HypVINN stats retained at:', HYPO_STATS)

## 10. Save the output to Drive and prepare the FRIENDS transfer

The subject directory is copied to Drive under `derivatives/fastsurfer/sub-136` and archived with that same relative path. The archive contains the processed subject directory; it excludes the uploaded input image, license, and Colab environment.

After downloading the archive from Drive, transfer it to M3 and extract it into a staging folder first. Review the new output before replacing the existing M3 `sub-136` folder, because the previous output was generated by FastSurfer 2.3 and this run uses v2.5.4.

In [ ]:
DRIVE_SUBJECT = RESULTS / 'derivatives' / 'fastsurfer' / SUBJECT
DRIVE_SUBJECT.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_SUBJECT.exists():
    raise FileExistsError(f'Results already exist at {DRIVE_SUBJECT}; use a fresh RUN_ID.')
shutil.copytree(OUTPUT, DRIVE_SUBJECT)

provenance.update({
    'completed_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'hypvinn_seg_sha256': sha256(HYPO_SEG),
    'hypvinn_stats_sha256': sha256(HYPO_STATS),
    'regional_volume_csv_sha256': sha256(RESULTS / 'regional_volumes_hard_label.csv'),
    'qc_status': 'needs_review',
    'output_relative_path': 'derivatives/fastsurfer/sub-136',
})
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))

ARCHIVE = RESULTS / f'FRIENDS_fastSurfer_sub-136_{RUN_ID}.tar.gz'
with tarfile.open(ARCHIVE, 'w:gz') as archive:
    archive.add(DRIVE_SUBJECT, arcname='derivatives/fastsurfer/sub-136')
print('Drive subject folder:', DRIVE_SUBJECT)
print('Transfer archive:', ARCHIVE)
print('Archive size (GiB):', round(ARCHIVE.stat().st_size / 1024**3, 3))

instructions = f"""# M3 transfer instructions

1. Download {ARCHIVE.name} from this Colab run folder in Google Drive.
2. Copy it to your FRIENDS project on M3, for example:
   scp {ARCHIVE.name} <your-M3-login>:/fs04/scratch2/gz87/FRIENDS/work/
3. On M3, unpack to a staging directory first:
   cd /fs04/scratch2/gz87/FRIENDS
   mkdir -p work/fastsurfer_colab_import/{RUN_ID}
   tar -xzf work/{ARCHIVE.name} -C work/fastsurfer_colab_import/{RUN_ID}
4. Review work/fastsurfer_colab_import/{RUN_ID}/derivatives/fastsurfer/sub-136.
5. After QC, run these commands from the FRIENDS project root to back up the old output and promote the new one:
   if [ -d derivatives/fastsurfer/sub-136 ]; then mv derivatives/fastsurfer/sub-136 work/fastsurfer_sub-136_backup_RUN_ID; fi
   mkdir -p derivatives/fastsurfer
   mv work/fastsurfer_colab_import/RUN_ID/derivatives/fastsurfer/sub-136 derivatives/fastsurfer/sub-136
   Replace RUN_ID in the two work paths with the run ID printed above.

The notebook cannot copy files directly to M3. The archive contains a complete FastSurfer subject directory with HypVINN and surfaces, and is laid out relative to the FRIENDS project root.
"""
(RESULTS / 'M3_transfer_instructions.md').write_text(instructions)
print('Saved run outputs, archive, and transfer instructions in:', RESULTS)

## Review checklist

- Confirm `derivatives/fastsurfer/sub-136/mri/hypothalamus.HypVINN.nii.gz` and `stats/hypothalamus.HypVINN.stats` are present.
- Inspect the whole-brain and HypVINN overlays; leave QC as `needs_review` until reviewed.
- Check Drive for the subject folder, logs, environment, CSV, provenance, archive, and M3 transfer instructions.
- On M3, extract to staging, review it, and back up the prior v2.3 output before replacing it.